# 🔤 Assamese OCR — Sentence Model Training (Kaggle)

This notebook trains a CRNN (CNN + LSTM) model for Assamese sentence-level OCR using CTC loss.

**Before you start:**
1. Set accelerator to **GPU** → Settings (right panel) → Accelerator → GPU T4 x2
2. Add your dataset containing `as-wiki-2021.txt` as a Kaggle Dataset input
3. (Optional) Add a dataset with your existing checkpoints `.pth` files

**Kaggle paths:**
- Input datasets: `/kaggle/input/<dataset-slug>/`
- Working directory (writable): `/kaggle/working/`
- Checkpoints will be saved to `/kaggle/working/checkpoints/`

---

## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL    = "https://github.com/rishav660/Assamese_OCR.git"
REPO_NAME   = "Assamese_OCR"
BRANCH      = "main"
WORK_DIR    = f'/kaggle/working/{REPO_NAME}/django_backend'

# Ensure /kaggle/working exists and shell cwd is valid (can get lost between cells)
os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

# Remove any existing clone and reclone fresh from the selected branch
if os.path.exists(f'/kaggle/working/{REPO_NAME}'):
    print('Removing stale clone...')
    !rm -rf /kaggle/working/{REPO_NAME}

print(f'Cloning {REPO_URL} branch {BRANCH}...')
!git clone -b {BRANCH} {REPO_URL} /kaggle/working/{REPO_NAME}

# Verify clone was successful before proceeding
if os.path.exists(WORK_DIR):
    # Switch to working directory
    %cd {WORK_DIR}
    !pwd
    print('✅ Successfully cloned and switched to working directory')
else:
    print(f'❌ Clone failed - {WORK_DIR} does not exist')
    print('Please check the git clone output above for errors')

In [ ]:
# Overwrite the cloned script with the fixed version that supports checkpoint resume
fixed_script = r"""
import argparse
import multiprocessing
import os
import time
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from char_map import char_to_idx, idx_to_char
from data_augmentation import get_training_transforms, get_validation_transforms
from dataset import AssameseOCRDataset, collate_fn
from metrics import OCRMetrics
from model import AssameseOCR


def default_num_workers():
    return 2 if os.name == "nt" else 4


def decode_ctc_greedy(log_probs, input_lengths, labels, target_lengths):
    """
    Decode CTC outputs with greedy search and reconstruct target strings.

    Args:
        log_probs:      (T, B, C) log-softmax output from the model
        input_lengths:  (B,) tensor of sequence lengths
        labels:         (sum(target_lengths),) flattened target indices
        target_lengths: (B,) tensor of target lengths per sample

    Returns:
        predictions: list[str] — decoded predictions
        targets:     list[str] — ground-truth strings
    """
    blank_idx = len(char_to_idx)
    batch_size = log_probs.size(1)

    # --- decode predictions ---
    preds = log_probs.permute(1, 0, 2).cpu()       # (B, T, C)
    preds = torch.argmax(preds, dim=2)              # (B, T)

    predictions = []
    for i in range(batch_size):
        seq = preds[i][: input_lengths[i]]
        prev = -1
        chars = []
        for idx in seq:
            idx = idx.item()
            if idx != prev and idx != blank_idx:
                ch = idx_to_char.get(idx, "")
                if ch:
                    chars.append(ch)
            prev = idx
        predictions.append("".join(chars))

    # --- reconstruct targets ---
    targets = []
    offset = 0
    for i in range(batch_size):
        length = target_lengths[i].item()
        indices = labels[offset : offset + length].cpu().tolist()
        target_str = "".join(idx_to_char.get(idx, "") for idx in indices)
        targets.append(target_str)
        offset += length

    return predictions, targets


def parse_args():
    parser = argparse.ArgumentParser(
        description="Train a sentence-level Assamese OCR model"
    )
    parser.add_argument(
        "--train-img-dir",
        type=str,
        default="data/train_real_sentences/images",
        help="Training image directory",
    )
    parser.add_argument(
        "--train-label-file",
        type=str,
        default="data/train_real_sentences/labels/labels.txt",
        help="Training labels manifest",
    )
    parser.add_argument(
        "--val-img-dir",
        type=str,
        default="data/val_real_sentences/images",
        help="Validation image directory",
    )
    parser.add_argument(
        "--val-label-file",
        type=str,
        default="data/val_real_sentences/labels/labels.txt",
        help="Validation labels manifest",
    )
    parser.add_argument(
        "--batch-size",
        type=int,
        default=32,
        help="Training batch size",
    )
    parser.add_argument(
        "--epochs",
        type=int,
        default=25,
        help="Maximum number of epochs",
    )
    parser.add_argument(
        "--learning-rate",
        type=float,
        default=0.0001,
        help="Initial learning rate",
    )
    parser.add_argument(
        "--weight-decay",
        type=float,
        default=0.01,
        help="AdamW weight decay",
    )
    parser.add_argument(
        "--scheduler-patience",
        type=int,
        default=3,
        help="ReduceLROnPlateau patience",
    )
    parser.add_argument(
        "--scheduler-factor",
        type=float,
        default=0.5,
        help="ReduceLROnPlateau decay factor",
    )
    parser.add_argument(
        "--early-stop-patience",
        type=int,
        default=5,
        help="Early stopping patience",
    )
    parser.add_argument(
        "--num-workers",
        type=int,
        default=default_num_workers(),
        help="DataLoader worker count",
    )
    parser.add_argument(
        "--disable-augmentation",
        action="store_true",
        help="Disable OCR-specific training augmentation",
    )
    parser.add_argument(
        "--best-checkpoint",
        type=str,
        default="checkpoints/best_model_sentences.pth",
        help="Path for best checkpoint",
    )
    parser.add_argument(
        "--final-checkpoint",
        type=str,
        default="checkpoints/final_model_sentences.pth",
        help="Path for final checkpoint",
    )
    parser.add_argument(
        "--latest-checkpoint",
        type=str,
        default="checkpoints/latest_model_sentences.pth",
        help="Path for latest epoch checkpoint",
    )
    parser.add_argument(
        "--resume",
        action="store_true",
        help="Resume training from the latest checkpoint if available",
    )
    parser.add_argument(
        "--resume-checkpoint",
        type=str,
        default=None,
        help="Optional checkpoint path to resume from (overrides --latest-checkpoint)",
    )
    parser.add_argument(
        "--plot-out",
        type=str,
        default="training_curve_sentences.png",
        help="Path for loss plot image",
    )
    return parser.parse_args()


def ensure_parent_dir(path_str):
    path = Path(path_str)
    if path.parent and str(path.parent) != ".":
        path.parent.mkdir(parents=True, exist_ok=True)


def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None, device=None):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        if optimizer is not None and "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if scheduler is not None and "scheduler_state_dict" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        return checkpoint
    model.load_state_dict(checkpoint)
    return {"epoch": -1}


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    if " " in char_to_idx:
        print(f"Space character found in char_map at index: {char_to_idx[' ']}")
    else:
        print("WARNING: space character is missing from char_map.")

    print("\nLoading sentence datasets...")
    print(f"  train images : {args.train_img_dir}")
    print(f"  train labels : {args.train_label_file}")
    print(f"  val images   : {args.val_img_dir}")
    print(f"  val labels   : {args.val_label_file}")

    train_transform = get_training_transforms(augment=not args.disable_augmentation)
    val_transform = get_validation_transforms()

    print(
        f"Data augmentation: {'disabled' if args.disable_augmentation else 'enabled'}"
    )

    train_dataset = AssameseOCRDataset(
        img_dir=args.train_img_dir,
        label_file=args.train_label_file,
        char_to_idx=char_to_idx,
        transform=train_transform,
    )

    val_dataset = AssameseOCRDataset(
        img_dir=args.val_img_dir,
        label_file=args.val_label_file,
        char_to_idx=char_to_idx,
        transform=val_transform,
    )

    print(f"Train sentences: {len(train_dataset)}, Val sentences: {len(val_dataset)}")

    if len(train_dataset) == 0:
        raise RuntimeError(
            "No training data found. Build train/val splits before starting training."
        )
    if len(val_dataset) == 0:
        raise RuntimeError(
            "No validation data found. Build a validation split before starting training."
        )

    pin_memory = torch.cuda.is_available()
    print(f"Using {args.num_workers} workers for data loading")

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=args.num_workers,
        pin_memory=pin_memory,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=args.num_workers,
        pin_memory=pin_memory,
    )

    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

    num_classes = len(char_to_idx) + 1
    model = AssameseOCR(img_height=64, nn_classes=num_classes).to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=args.learning_rate,
        weight_decay=args.weight_decay,
    )
    criterion = nn.CTCLoss(
        blank=len(char_to_idx),
        reduction="mean",
        zero_infinity=True,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        "min",
        patience=args.scheduler_patience,
        factor=args.scheduler_factor,
    )

    start_epoch = 0
    best_val_loss = float("inf")
    best_cer = float("inf")
    no_improve_epochs = 0

    if args.resume:
        resume_path = args.resume_checkpoint or args.latest_checkpoint
        if resume_path and os.path.exists(resume_path):
            print(f"Resuming from checkpoint: {resume_path}")
            checkpoint = load_checkpoint(
                resume_path,
                model,
                optimizer,
                scheduler,
                device,
            )
            if checkpoint is not None and isinstance(checkpoint, dict):
                start_epoch = checkpoint.get("epoch", -1) + 1
                best_val_loss = checkpoint.get("best_val_loss", best_val_loss)
                best_cer = checkpoint.get("best_cer", best_cer)
                no_improve_epochs = checkpoint.get("no_improve_epochs", no_improve_epochs)
                print(f"Resuming training at epoch {start_epoch + 1}")
            else:
                print("Loaded model weights only; starting from epoch 1.")
        else:
            print(f"Resume requested but checkpoint not found: {resume_path}")

    print("\n" + "=" * 60)
    print("Sentence OCR training")
    print("=" * 60)
    print(f"Epochs: {args.epochs}")
    print(f"Batch size: {args.batch_size}")
    print(f"Learning rate: {args.learning_rate}")
    print(f"Weight decay: {args.weight_decay}")
    print("=" * 60 + "\n")

    print("Loading first batch for sanity check...")
    sample_batch = next(iter(train_loader))
    if sample_batch is not None:
        images, labels, input_lengths, target_lengths = sample_batch
        print(f"Batch loaded: {images.shape}")
        print(f"Sequence length: {input_lengths[0].item()}")
        print(f"Sample target lengths: {target_lengths[:4].tolist()}")
        space_idx = char_to_idx.get(" ", -1)
        print(f"Labels contain spaces: {space_idx > 0 and space_idx in labels}")

    best_val_loss = float("inf")
    best_cer = float("inf")
    train_losses = []
    val_losses = []
    val_cers = []
    val_wers = []
    no_improve_epochs = 0

    ensure_parent_dir(args.best_checkpoint)
    ensure_parent_dir(args.final_checkpoint)
    ensure_parent_dir(args.latest_checkpoint)
    ensure_parent_dir(args.plot_out)

    print("\nStarting training...\n")

    epoch = start_epoch - 1
    try:
        for epoch in range(start_epoch, args.epochs):
            model.train()
            total_loss = 0.0
            batch_count = 0
            epoch_start = time.time()

            for batch_idx, batch in enumerate(train_loader):
                if batch is None:
                    continue

                images, labels, input_lengths, target_lengths = batch
                images = images.to(device)
                labels = labels.to(device)
                input_lengths = input_lengths.to(device)
                target_lengths = target_lengths.to(device)

                outputs = model(images)
                outputs = torch.log_softmax(outputs, 2)
                loss = criterion(outputs, labels, input_lengths, target_lengths)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

                total_loss += loss.item()
                batch_count += 1

                if batch_idx % 50 == 0:
                    elapsed = time.time() - epoch_start
                    batches_per_sec = (batch_idx + 1) / elapsed if elapsed > 0 else 0
                    eta = (
                        (len(train_loader) - batch_idx) / batches_per_sec
                        if batches_per_sec > 0
                        else 0
                    )
                    print(
                        f"Epoch {epoch + 1}/{args.epochs} | "
                        f"Batch {batch_idx}/{len(train_loader)} | "
                        f"Loss: {loss.item():.4f} | "
                        f"Speed: {batches_per_sec:.2f} batch/s | ETA: {eta:.0f}s"
                    )

            avg_train_loss = total_loss / batch_count if batch_count > 0 else 0.0
            train_losses.append(avg_train_loss)

            model.eval()
            val_loss = 0.0
            val_batch_count = 0
            val_metrics = OCRMetrics()

            with torch.no_grad():
                for batch in val_loader:
                    if batch is None:
                        continue

                    images, labels, input_lengths, target_lengths = batch
                    images = images.to(device)
                    labels = labels.to(device)
                    input_lengths = input_lengths.to(device)
                    target_lengths = target_lengths.to(device)

                    outputs = model(images)
                    outputs = torch.log_softmax(outputs, 2)
                    loss = criterion(outputs, labels, input_lengths, target_lengths)

                    val_loss += loss.item()
                    val_batch_count += 1

                    predictions, targets = decode_ctc_greedy(
                        outputs, input_lengths, labels, target_lengths
                    )
                    val_metrics.update(predictions, targets)

            avg_val_loss = val_loss / val_batch_count if val_batch_count > 0 else 0.0
            val_losses.append(avg_val_loss)
            val_cers.append(val_metrics.cer)
            val_wers.append(val_metrics.wer)
            epoch_time = time.time() - epoch_start

            print(f"\n{'=' * 60}")
            print(
                f"Epoch {epoch + 1}/{args.epochs} completed in {epoch_time:.1f}s "
                f"({epoch_time / 60:.1f} min)"
            )
            print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
            print(f"Val {val_metrics.summary()}")
            print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_cer = val_metrics.cer
                torch.save(model.state_dict(), args.best_checkpoint)
                print(f"Best model saved to: {args.best_checkpoint}")
                no_improve_epochs = 0
            else:
                no_improve_epochs += 1
                print(f"No improvement for {no_improve_epochs} epoch(s)")

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_val_loss": best_val_loss,
                    "best_cer": best_cer,
                    "no_improve_epochs": no_improve_epochs,
                },
                args.latest_checkpoint,
            )
            print(f"Latest checkpoint saved to: {args.latest_checkpoint}")

            print(f"{'=' * 60}\n")

            scheduler.step(avg_val_loss)

            if no_improve_epochs >= args.early_stop_patience:
                print(f"Early stopping after {epoch + 1} epochs")
                break

    except KeyboardInterrupt:
        print("\nKeyboardInterrupt detected. Saving latest checkpoint before exit...")
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_loss": best_val_loss,
                "best_cer": best_cer,
                "no_improve_epochs": no_improve_epochs,
            },
            args.latest_checkpoint,
        )
        print(f"Latest checkpoint saved to: {args.latest_checkpoint}")
        return

    torch.save(model.state_dict(), args.final_checkpoint)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    ax1.plot(train_losses, label="Training Loss")
    ax1.plot(val_losses, label="Validation Loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.set_title("Loss")

    ax2.plot(val_cers, label="CER", marker="o", markersize=3)
    ax2.plot(val_wers, label="WER", marker="s", markersize=3)
    ax2.set_xlabel("Epochs")
    ax2.set_ylabel("Error Rate")
    ax2.legend()
    ax2.set_title("Character & Word Error Rate")

    fig.suptitle("Sentence Model Training", fontsize=14)
    fig.tight_layout()
    fig.savefig(args.plot_out)

    print("\n" + "=" * 60)
    print("Training complete")
    print("=" * 60)
    print(f"Best validation loss: {best_val_loss:.4f}")
    print(f"Best CER at that point: {best_cer:.4f} ({best_cer * 100:.2f}%)")
    print(
        f"Final CER: {val_cers[-1]:.4f} ({val_cers[-1] * 100:.2f}%)"
        if val_cers
        else ""
    )
    print(
        f"Final WER: {val_wers[-1]:.4f} ({val_wers[-1] * 100:.2f}%)"
        if val_wers
        else ""
    )
    print(f"Best checkpoint: {args.best_checkpoint}")
    print(f"Final checkpoint: {args.final_checkpoint}")
    print(f"Loss plot: {args.plot_out}")
    print("=" * 60)


if __name__ == "__main__":
    multiprocessing.freeze_support()
    train(parse_args())
"""

with open('train_sentence_model.py', 'w', encoding='utf-8') as f:
    f.write(fixed_script)
print('✅ Overwrote train_sentence_model.py with the fixed resume-capable version')

In [ ]:
# Verify the cloned script supports --resume (guards against stale cache)
import subprocess
result = subprocess.run(
    ['python', 'train_sentence_model.py', '--help'],
    capture_output=True, text=True
)
help_text = result.stdout + result.stderr
if '--resume' in help_text and '--latest-checkpoint' in help_text:
    print('✅ Script version OK — --resume and --latest-checkpoint supported')
else:
    print('❌ Script is outdated — re-run the clone cell above before continuing')
    print(help_text[:500])

In [ ]:
# Install dependencies
!pip install -q -r requirements-train.txt

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Enable GPU in Settings → Accelerator.')

## 2. Locate Input Data & Set Up Directories

Your `as-wiki-2021.txt` must be added as a Kaggle Dataset.

**How to add it:**
1. Upload `as-wiki-2021.txt` to a Kaggle Dataset (or use an existing one)
2. In this notebook → top-right → **+ Add Data** → search for your dataset
3. It will appear at `/kaggle/input/<your-dataset-slug>/as-wiki-2021.txt`

Update `WIKI_INPUT_PATH` below to match your dataset slug.

In [ ]:
import os
import shutil

# ─── UPDATE THIS to match your Kaggle dataset slug ───────────────────────────
WIKI_INPUT_PATH = '/kaggle/input/assamese-wiki/as-wiki-2021.txt'  # ← CHANGE THIS
# ─────────────────────────────────────────────────────────────────────────────

# Verify the wiki file exists
if os.path.exists(WIKI_INPUT_PATH):
    size_mb = os.path.getsize(WIKI_INPUT_PATH) / 1e6
    print(f'✅ Wiki file found: {WIKI_INPUT_PATH} ({size_mb:.1f} MB)')
else:
    print(f'❌ Wiki file not found at: {WIKI_INPUT_PATH}')
    print('   Add your dataset via "+ Add Data" in the right panel.')

# Create local data/ and checkpoints/ directories under the working dir
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# Symlink wiki file into data/ for convenience
wiki_link = 'data/as-wiki-2021.txt'
if not os.path.exists(wiki_link):
    os.symlink(WIKI_INPUT_PATH, wiki_link)
    print(f'✅ Symlinked wiki file → data/as-wiki-2021.txt')
else:
    print(f'✅ data/as-wiki-2021.txt already linked')

In [ ]:
# (Optional) Copy existing checkpoints from a Kaggle dataset input
# Add your checkpoint dataset via "+ Add Data", then update the path below.

CHECKPOINT_INPUT = '/kaggle/input/assamese-ocr-checkpoints'  # ← CHANGE or comment out

if os.path.exists(CHECKPOINT_INPUT):
    for f in os.listdir(CHECKPOINT_INPUT):
        if f.endswith('.pth'):
            src = os.path.join(CHECKPOINT_INPUT, f)
            dst = os.path.join('checkpoints', f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f'  Copied: {f}')
            else:
                print(f'  Already exists: {f}')
    print('✅ Checkpoints ready')
else:
    print('ℹ️  No checkpoint input dataset found — training from scratch.')

print('\nCheckpoints directory:')
!ls -lh checkpoints/ 2>/dev/null || echo '  (empty)'

## 3. Generate Non-Overlapping Train / Val / Test Splits

In [ ]:
# Check if splits already exist (useful if re-running the notebook)
train_labels = 'data/train_real_sentences/labels/labels.txt'
if os.path.exists(train_labels):
    with open(train_labels) as f:
        n = sum(1 for _ in f)
    print(f'✅ Splits already exist! ({n} training samples)')
    print('Skip the next cell unless you want to regenerate.')
else:
    print('❌ No splits found. Run the next cell to generate them.')

In [ ]:
# Step 1: Clean up any old split directories
for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    dst = f'data/{split}'
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.makedirs(f'{dst}/images', exist_ok=True)
    os.makedirs(f'{dst}/labels', exist_ok=True)

print('✅ Local data directories ready')

# Step 2: Generate splits
!python build_real_sentence_splits.py \
    --input data/as-wiki-2021.txt \
    --train-output data/train_real_sentences \
    --val-output data/val_real_sentences \
    --test-output data/test_real_sentences \
    --train-count 50000 \
    --val-count 5000 \
    --test-count 2000 \
    --seed 42

# Step 3: Verify generated splits
print()
for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    img_dir    = f'data/{split}/images'
    label_file = f'data/{split}/labels/labels.txt'
    n_images   = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    ok         = '✅' if os.path.exists(label_file) else '❌'
    print(f'{split}')
    print(f'  Images     : {n_images}')
    print(f'  labels.txt : {ok}')

print('\n✅ Dataset generation complete')

## 4. Train the Sentence Model 🚀

In [ ]:
# Main training run
!python train_sentence_model.py \
    --train-img-dir   data/train_real_sentences/images \
    --train-label-file data/train_real_sentences/labels/labels.txt \
    --val-img-dir     data/val_real_sentences/images \
    --val-label-file  data/val_real_sentences/labels/labels.txt \
    --epochs 25 \
    --batch-size 32 \
    --learning-rate 0.0001 \
    --best-checkpoint  checkpoints/best_model_sentences.pth \
    --final-checkpoint checkpoints/final_model_sentences.pth \
    --latest-checkpoint checkpoints/latest_model_sentences.pth \
    --resume \
    --plot-out training_curve_sentences.png

In [ ]:
# View training curve
from IPython.display import Image as IPImage, display
if os.path.exists('training_curve_sentences.png'):
    display(IPImage('training_curve_sentences.png'))
else:
    print('No training curve found yet.')

## 5. Evaluate Model Accuracy (CER / WER / Exact Match)

In [ ]:
# Evaluate on VALIDATION set — raw model output (greedy decode)
!python evaluate.py \
    --img-dir    data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --num-samples 10

In [ ]:
# Evaluate on VALIDATION set — with spell-check post-processing
# Compare with the cell above to see if post-processing helps
!python evaluate.py \
    --img-dir    data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --post-process \
    --num-samples 10

In [ ]:
# Evaluate on VALIDATION set — CTC Beam Search (typically better accuracy)
!python evaluate.py \
    --img-dir    data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --beam-width 10 \
    --num-samples 10

In [ ]:
# Evaluate on TEST set
test_labels = 'data/test_real_sentences/labels/labels.txt'
if os.path.exists(test_labels):
    !python evaluate.py \
        --img-dir    data/test_real_sentences/images \
        --label-file {test_labels} \
        --checkpoint checkpoints/best_model_sentences.pth \
        --num-samples 10 \
        --output test_results.tsv
    print('\n✅ Detailed results saved to test_results.tsv')
else:
    print('❌ No test set found. Regenerate splits with --test-count 2000.')

In [ ]:
# Quick single-image prediction with ground-truth comparison
!python predict_cli.py \
    --image        data/val_real_sentences/images/sentence_000000.png \
    --checkpoint   checkpoints/best_model_sentences.pth \
    --ground-truth data/val_real_sentences/labels/sentence_000000.txt

## 6. (Optional) Transfer Learning

If you have `best_model_fast.pth` in `checkpoints/`, you can fine-tune from the character model.

In [ ]:
# Transfer learning — uncomment to run
# !python train_transfer_learning.py

## 7. Save Outputs

On Kaggle, anything written to `/kaggle/working/` is automatically saved as notebook output when the session ends.  
You can download your checkpoints, results, and plots from the **Output** tab after the run completes.

In [ ]:
# List all saved outputs
print('=== Checkpoints ===')
!ls -lh checkpoints/ 2>/dev/null || echo '  (none)'

print('\n=== Plots & Results ===')
!ls -lh *.png *.tsv 2>/dev/null || echo '  (none)'

print()
print('✅ All files in /kaggle/working/ are available in the Output tab.')

In [ ]:
# (Optional) Push code changes back to GitHub
# Uncomment and fill in your credentials / token if needed

# import subprocess
# GITHUB_TOKEN = 'your_token_here'  # Use a Kaggle Secret instead
# repo_dir = f'/kaggle/working/{REPO_NAME}'
# subprocess.run(['git', '-C', repo_dir, 'add', '-A'])
# subprocess.run(['git', '-C', repo_dir, 'commit', '-m', 'Kaggle training updates'])
# subprocess.run(['git', '-C', repo_dir, 'push', 'origin', 'develop'])